In [41]:
# notre jeu de donnee
import pandas as pd
path = 'twitter_training.csv'
df = pd.read_csv(path, header=None, 
                 names=['id', 'entity', 'sentiment', 'content'])

df_select = df[['content']].head(1000).copy()
df_select

,content
0,im getting on borderlands and i will murder yo...
1,I am coming to the borders and I will kill you...
2,im getting on borderlands and i will kill you ...
3,im coming on borderlands and i will murder you...
4,im getting on borderlands 2 and i will murder ...
...,...
995,of
996,Who's down for some @Borderlands on
997,Who's on for some @ Borderlands
998,Who's at @ Borderlands


In [42]:
# la phase de nettoyage de notre jeu de donnee(preprocessing)
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import re
import string

# Télécharger les ressources nécessaires (à exécuter une seule fois)
# nltk.download('punkt')
# nltk.download('stopwords')
# nltk.download('wordnet')
# nltk.download('omw-1.4')

# Initialisation
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocessing(text):
    # Vérifier que le texte est bien une chaîne
    if not isinstance(text, str):
        return ""
    
    # Suppression de la ponctuation et des caractères spéciaux
    text = re.sub(r"[^\w\s]", "", text)
    
    # Conversion en minuscules
    text = text.lower()
    
    # Tokenisation
    tokens = word_tokenize(text)
    
    # Suppression des stopwords et lemmatisation
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words and word not in string.punctuation
    ]
    
    # Reconstruction du texte
    clean_text = ' '.join(tokens)
    return clean_text


df_select['content_nettoye'] =df_select['content'].apply(preprocessing)

# Afficher le résultat
df_select.head(1000)

,content,content_nettoye
0,im getting on borderlands and i will murder yo...,im getting borderland murder
1,I am coming to the borders and I will kill you...,coming border kill
2,im getting on borderlands and i will kill you ...,im getting borderland kill
3,im coming on borderlands and i will murder you...,im coming borderland murder
4,im getting on borderlands 2 and i will murder ...,im getting borderland 2 murder
...,...,...
995,of,
996,Who's down for some @Borderlands on,who borderland
997,Who's on for some @ Borderlands,who borderland
998,Who's at @ Borderlands,who borderland


In [51]:
# la phase de transformation de nos donnee en version numerique(feature extraction)
import numpy as np
import pandas as pd
import re
from collections import Counter
import math

class SparseMatrixSim:
    """Classe utilitaire pour simuler le comportement de scikit-learn"""
    def __init__(self, data):
        self.data = data
        self.shape = data.shape

    def toarray(self):
        return self.data

class CustomTFIDF:
    def __init__(self):
        self.vocabulary = {}
        self.idf_vector = {}

    def _tokenize(self, text):
        return re.findall(r"(?u)\b\w\w+\b", str(text).lower())

    def fit(self, corpus):
        num_documents = len(corpus)
        df_counts = Counter()
        
        for doc in corpus:
            tokens = set(self._tokenize(doc))
            for token in tokens:
                df_counts[token] += 1
        
        sorted_words = sorted(df_counts.keys())
        self.vocabulary = {word: i for i, word in enumerate(sorted_words)}
        
        for word in self.vocabulary:
            count = df_counts[word]
            # Formule Smooth IDF (sklearn)
            self.idf_vector[word] = math.log((1 + num_documents) / (1 + count)) + 1
        
        return self

    def transform(self, corpus):
        rows = []
        vocab_size = len(self.vocabulary)
        
        for doc in corpus:
            tokens = self._tokenize(doc)
            tf_counts = Counter(tokens)
            vector = np.zeros(vocab_size)
            
            for word, count in tf_counts.items():
                if word in self.vocabulary:
                    vector[self.vocabulary[word]] = count * self.idf_vector[word]
            
            # Normalisation L2
            norm = np.linalg.norm(vector)
            if norm > 0:
                vector = vector / norm
            rows.append(vector)
            
        # On retourne l'objet simulateur pour permettre l'appel à .toarray()
        return SparseMatrixSim(np.array(rows))

    def fit_transform(self, corpus):
        return self.fit(corpus).transform(corpus)

    def get_feature_names_out(self):
        return np.array([word for word, index in sorted(self.vocabulary.items(), key=lambda item: item[1])])



# 
tfidf_engine = CustomTFIDF()
features_obj = tfidf_engine.fit_transform(df_select['content_nettoye'])

# Maintenant vous pouvez faire comme avec sklearn !
features_array = features_obj.toarray()
print(tfidf_engine.get_feature_names_out())
print(features_array)

print("Dimensions de la matrice :", features_array.shape)

['02' '0sku6vr4ixu' '10' ... 'youve' 'zelda' 'zer0']
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
Dimensions de la matrice : (1000, 1912)


In [53]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Créer une instance du vectoriseur TfidfVectorizer
vectorizer = TfidfVectorizer()

# Appliquer le vectoriseur sur la colonne 'texte_nettoye'
features = vectorizer.fit_transform(df_select['content_nettoye'])

# Convertir les caractéristiques en une représentation de matrice creuse
features = features.toarray()
print(features)
print(vectorizer.get_feature_names_out())
# Afficher les dimensions de la matrice de caractéristiques
print("Dimensions de la matrice de caractéristiques :", features.shape)

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
['02' '0sku6vr4ixu' '10' ... 'youve' 'zelda' 'zer0']
Dimensions de la matrice de caractéristiques : (1000, 1912)
